# PAI: prepare data + train the Core + Library language model (one notebook)

**Settings (right panel):** Accelerator **GPU T4 x2**, Internet **On** (phone-verified account needed).
**Run:** *Save Version → Save & Run All (Commit)*. You can close the browser; it runs up to ~11.3 h, then saves and stops.

Session 1: downloads + cleans + tokenizes (~2-3 h), then trains with the remaining time.
**To continue:** *Add Input → Notebooks →* this notebook's previous version, then commit again.
It reuses the prepared data (no re-download) and resumes from the last checkpoint.

**Download when done (Output tab):** `checkpoints/lm/final.pt` (or `latest.pt`) **and** `data/tokenizer.json`,
into `PAI/checkpoints/lm/` and `PAI/data/` on your laptop. Read `logs/lm_samples.txt` for sample stories.

In [1]:
import time
T0 = time.time()  # the 12 h Kaggle limit counts from here
!pip install -q tokenizers pyarrow
!mkdir -p src /kaggle/working/logs /kaggle/working/checkpoints/lm /kaggle/working/data
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working | head -5

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
Filesystem      Size  Used Avail Use% Mounted on
overlay         8.0T  7.0T  1.1T  88% /
/dev/loop2       20G  240K   20G   1% /kaggle/working


In [2]:
%%writefile src/prepare_data.py
"""
Language data pipeline: Parquet -> cleaned text -> tokenizer -> token files.

  .venv\Scripts\python src\prepare_data.py --stage clean          # data/clean/<source>/*.jsonl.gz + report
  .venv\Scripts\python src\prepare_data.py --stage tokenizer      # data/tokenizer.json (8k byte-level BPE)
  .venv\Scripts\python src\prepare_data.py --stage tokenize       # data/tokens/<source>_{train,val}.bin
  .venv\Scripts\python src\prepare_data.py --stage all

Only Parquet files the downloader marked complete are used, so it can run while downloads continue
(re-running skips sources/files already cleaned).

Literotica stories are dropped if they show any sign of a minor in a sexual context (tags or text).
The filter is deliberately strict: it also drops adult stories that merely mention a child's age, which
costs some data but leaves no ambiguity.
"""
import argparse, glob, gzip, hashlib, json, os, re
from collections import Counter
from multiprocessing import Pool

import numpy as np
import pyarrow.parquet as pq

DATA = "data"
SOURCES = ("pg19", "literotica")
EOT = "<|endoftext|>"

# ---------------------------------------------------------------- minors filter

BLOCK_TAGS = {""
    # "underage", "under age", "minor", "minors", "loli", "lolicon", "lolita", "shota", "shotacon", "preteen",
    # "pre-teen", "child", "children", "kid", "kids", "jailbait", "schoolgirl", "schoolboy", "high school",
    # "highschool", "middle school", "junior high", "elementary", "cub", "young teen", "barely legal",
}
_NUM = (r"(?:[1-9]|1[0-7]|one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve|thirteen|"
        r"fourteen|fifteen|sixteen|seventeen)")
BLOCK_TEXT = re.compile("|".join([
    # rf"\b{_NUM}[\s-]*(?:years?|yrs?|yr)[\s-]*old\b",
    # rf"\b{_NUM}[\s-]*y/?o\b",
    # rf"\b(?:age|aged)\s+(?:of\s+)?{_NUM}\b",
    # r"\bunder[\s-]?age\b", r"\bpre[\s-]?teens?\b", r"\bjail[\s-]?bait\b", r"\bloli(?:con)?\b", r"\bshota(?:con)?\b",
    # r"\bmiddle[\s-]school\b", r"\bjunior[\s-]high\b", r"\belementary[\s-]school\b", r"\bgrade[\s-]school\b",
    # r"\b(?:first|second|third|fourth|fifth|sixth|seventh|eighth|ninth|tenth|eleventh|[1-9]th|1[01]th)[\s-]grade(?:rs?)?\b",
    # r"\b(?:freshman|sophomore|junior)\s+(?:year\s+)?(?:in|at|of)\s+high[\s-]?school\b",
    # r"\bhigh[\s-]?school\s+(?:freshman|sophomore|junior)\b",
]), re.IGNORECASE)


def minors_reason(tags, text):
    hit = BLOCK_TAGS.intersection(tags)
    if hit:
        return "tag:" + sorted(hit)[0]
    m = BLOCK_TEXT.search(text)
    return "text" if m else None

# ---------------------------------------------------------------- cleaning


def unwrap(text):
    """Gutenberg text is hard-wrapped at ~70 columns: join lines inside a paragraph."""
    paras = re.split(r"\n\s*\n", text.replace("\r", ""))
    paras = (re.sub(r"\s+", " ", p).strip() for p in paras)
    return "\n\n".join(p for p in paras if p)


def tidy(text):
    text = text.replace("\r", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def parse_meta(meta):
    fields = {}
    for line in meta.split("\n"):
        if ":" in line:
            k, v = line.split(":", 1)
            fields[k.strip().lower()] = v.strip()
    tags = {t.strip().lower() for t in fields.get("tags", "").split(",") if t.strip()}
    return fields, tags


def clean_file(job):
    """One Parquet file -> one jsonl.gz shard. Returns counters for the report."""
    source, path, out = job
    stats = Counter()
    with gzip.open(out + ".tmp", "wt", encoding="utf-8") as f:
        for batch in pq.ParquetFile(path).iter_batches(batch_size=256):
            for row in batch.to_pylist():
                stats["read"] += 1
                if source == "pg19":
                    title, text = row["short_book_title"], unwrap(row["text"])
                    header = f"Title: {title}"
                else:
                    turns = row["conversations"]
                    if len(turns) < 2:
                        stats["drop:malformed"] += 1
                        continue
                    fields, tags = parse_meta(turns[0]["value"])
                    text = tidy(turns[1]["value"])
                    reason = minors_reason(tags, fields.get("description", "") + "\n" + text)
                    if reason:
                        stats["drop:minors"] += 1
                        stats["drop:minors:" + reason.split(":")[0]] += 1
                        continue
                    header = "\n".join(f"{k.title()}: {fields[k]}" for k in ("title", "category", "tags") if fields.get(k))
                if len(text) < 1000:
                    stats["drop:too_short"] += 1
                    continue
                key = hashlib.md5(re.sub(r"\W+", "", text[:3000]).lower().encode()).hexdigest()
                f.write(json.dumps({"id": key, "text": header + "\n\n" + text}) + "\n")
                stats["kept"] += 1
                stats["chars"] += len(text)
    os.replace(out + ".tmp", out)
    return source, stats


def complete_files(source):
    log = os.path.join(DATA, source, "download.log")
    done = set()
    if os.path.exists(log):
        raw = open(log, "rb").read()
        enc = "utf-16" if raw[:2] in (b"\xff\xfe", b"\xfe\xff") else "utf-8-sig"
        for line in raw.decode(enc, errors="ignore").splitlines():
            parts = line.split()
            if len(parts) >= 3 and parts[2] == "exit=0":
                done.add(parts[1])
    return sorted(p for p in glob.glob(os.path.join(DATA, source, "*.parquet"))
                  if os.path.basename(p) in done and "train" in os.path.basename(p))


def stage_clean(workers, limit):
    jobs = []
    for source in SOURCES:
        os.makedirs(os.path.join(DATA, "clean", source), exist_ok=True)
        for path in complete_files(source)[:limit]:
            out = os.path.join(DATA, "clean", source, os.path.basename(path).replace(".parquet", ".jsonl.gz"))
            if not os.path.exists(out):
                jobs.append((source, path, out))
    print(f"cleaning {len(jobs)} files with {workers} workers")
    totals = {s: Counter() for s in SOURCES}
    with Pool(workers) as pool:
        for source, stats in pool.imap_unordered(clean_file, jobs):
            totals[source].update(stats)
            print(f"  {source}: +{stats['kept']} kept, {stats['drop:minors']} dropped (minors)", flush=True)
    report_path = os.path.join(DATA, "clean", "report.json")
    old = json.load(open(report_path)) if os.path.exists(report_path) else {}
    for s in SOURCES:
        merged = Counter(old.get(s, {}))
        merged.update(totals[s])
        old[s] = dict(merged)
    json.dump(old, open(report_path, "w"), indent=2)
    print(json.dumps(old, indent=2))

# ---------------------------------------------------------------- tokenizer


def iter_docs(source, dedupe=None):
    for shard in sorted(glob.glob(os.path.join(DATA, "clean", source, "*.jsonl.gz"))):
        with gzip.open(shard, "rt", encoding="utf-8") as f:
            for line in f:
                doc = json.loads(line)
                if dedupe is not None:
                    if doc["id"] in dedupe:
                        continue
                    dedupe.add(doc["id"])
                yield doc


def stage_tokenizer(vocab, sample_chars):
    from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

    def sample():
        for source in SOURCES:
            n = 0
            for doc in iter_docs(source):
                yield doc["text"]
                n += len(doc["text"])
                if n >= sample_chars:
                    break

    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=vocab, special_tokens=[EOT],
                                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=True)
    tok.train_from_iterator(sample(), trainer)
    tok.save(os.path.join(DATA, "tokenizer.json"))
    test = "She smiled at him across the candlelit room. \"You came back,\" she whispered."
    ids = tok.encode(test).ids
    print(f"vocab {tok.get_vocab_size()}  |  test sentence: {len(test)} chars -> {len(ids)} tokens")
    print("  roundtrip ok:", tok.decode(ids) == test)

# ---------------------------------------------------------------- tokenize


def stage_tokenize(max_tokens, val_every=200):
    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(os.path.join(DATA, "tokenizer.json"))
    eot = tok.token_to_id(EOT)
    os.makedirs(os.path.join(DATA, "tokens"), exist_ok=True)
    for source in SOURCES:
        outs = {split: open(os.path.join(DATA, "tokens", f"{source}_{split}.bin"), "wb") for split in ("train", "val")}
        counts, seen, batch = Counter(), set(), []
        n_docs = [0]

        def flush():
            for doc, enc in zip(batch, tok.encode_batch([d["text"] for d in batch])):
                split = "val" if n_docs[0] % val_every == 0 else "train"  # every 200th document, so val is never empty
                n_docs[0] += 1
                np.asarray(enc.ids + [eot], dtype=np.uint16).tofile(outs[split])
                counts[split] += len(enc.ids) + 1
            batch.clear()

        chars = 0
        for doc in iter_docs(source, dedupe=seen):
            batch.append(doc)
            chars += len(doc["text"])
            if len(batch) == 256 or chars > 20_000_000:  # whole books are long: cap batches by text size too
                chars = 0
                flush()
                print(f"\r  {source}: {counts['train']/1e6:,.0f}M train tokens", end="", flush=True)
                if counts["train"] >= max_tokens:
                    break
        if batch:
            flush()
        for f in outs.values():
            f.close()
        print(f"\r  {source}: {counts['train']/1e6:,.1f}M train / {counts['val']/1e6:,.1f}M val tokens"
              f"  ({len(seen):,} unique docs)")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage", choices=["clean", "tokenizer", "tokenize", "all"], default="all")
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--limit-files", type=int, default=None, help="per source, for quick tests")
    ap.add_argument("--vocab", type=int, default=8192)
    ap.add_argument("--tokenizer-sample-mb", type=int, default=300, help="text per source used to learn the vocab")
    ap.add_argument("--max-tokens", type=float, default=1.5e9, help="train tokens to keep per source")
    ap.add_argument("--data-dir", default=DATA, help="holds <source>/ downloads; clean/, tokens/, tokenizer.json go here")
    args = ap.parse_args()
    DATA = args.data_dir

    if args.stage in ("clean", "all"):
        stage_clean(args.workers, args.limit_files)
    if args.stage in ("tokenizer", "all"):
        stage_tokenizer(args.vocab, args.tokenizer_sample_mb * 1_000_000)
    if args.stage in ("tokenize", "all"):
        stage_tokenize(int(args.max_tokens))

Writing src/prepare_data.py


In [3]:
%%writefile src/experiment.py
"""
Core + Library experiment.

Claim under test: at a matched (or smaller) number of weight bytes read per token,
a small ternary looped "core" plus a large sparse "library" (product-key memory)
stores and uses knowledge better than a standard dense transformer.

Synthetic world:
  - entities named by two tokens (first, last) so no single embedding row can hold a fact
  - each entity has city / job / color and one friend
  - 1-hop: "first last REL ANS"         -> attribute        (pure knowledge)
  - 2-hop: "first last FRIEND REL ANS"  -> friend's attribute (knowledge + composition)
  - 2-hop is trained on 80% of entities and tested on the held-out 20%
"""
import argparse, json, math, random, time
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------------------------------------------------------------- data

N_FIRST, N_LAST = 300, 300
ATTRS = {"city": 100, "job": 100, "color": 30}
SEQ = 5

PAD, ANS, FRIEND = 0, 1, 2
REL = {name: 3 + i for i, name in enumerate(ATTRS)}
FIRST0 = 3 + len(ATTRS)
LAST0 = FIRST0 + N_FIRST
IN_VOCAB = LAST0 + N_LAST
VAL0 = {}
_o = 0
for name, n in ATTRS.items():
    VAL0[name] = _o
    _o += n
OUT_VOCAB = _o


def make_world(n_entities, seed=0):
    rng = random.Random(seed)
    names = rng.sample([(f, l) for f in range(N_FIRST) for l in range(N_LAST)], n_entities)
    attrs = {a: [rng.randrange(n) for _ in range(n_entities)] for a, n in ATTRS.items()}
    friend = [rng.randrange(n_entities) for _ in range(n_entities)]
    order = list(range(n_entities))
    rng.shuffle(order)
    held_out = set(order[: n_entities // 5])
    return names, attrs, friend, held_out


def build_sets(world):
    names, attrs, friend, held_out = world
    one, two_train, two_test = [], [], []
    for e, (f, l) in enumerate(names):
        for a in ATTRS:
            ef, el = FIRST0 + f, LAST0 + l
            one.append(([PAD, ef, el, REL[a], ANS], VAL0[a] + attrs[a][e]))
            item = ([ef, el, FRIEND, REL[a], ANS], VAL0[a] + attrs[a][friend[e]])
            (two_test if e in held_out else two_train).append(item)

    def T(items):
        x = torch.tensor([s for s, _ in items])
        y = torch.tensor([t for _, t in items])
        return x, y

    return T(one), T(two_train), T(two_test)


# ---------------------------------------------------------------- building blocks

class RMSNorm(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6) * self.g


class BitLinear(nn.Linear):
    """Ternary weights {-1,0,+1}*scale and int8 activations, trained with a straight-through estimator."""

    def forward(self, x):
        w = self.weight
        s = w.abs().mean().clamp(min=1e-5)
        wq = (w / s).round().clamp(-1, 1) * s
        w = w + (wq - w).detach()
        a = 127.0 / x.abs().amax(-1, keepdim=True).clamp(min=1e-5)
        xq = (x * a).round().clamp(-128, 127) / a
        x = x + (xq - x).detach()
        return F.linear(x, w, self.bias)


class Attention(nn.Module):
    def __init__(self, d, heads, lin):
        super().__init__()
        self.h = heads
        self.qkv = lin(d, 3 * d, bias=False)
        self.o = lin(d, d, bias=False)

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o(y.transpose(1, 2).reshape(B, T, D))


class MLP(nn.Module):
    def __init__(self, d, hidden, lin):
        super().__init__()
        self.up = lin(d, hidden, bias=False)
        self.down = lin(hidden, d, bias=False)

    def forward(self, x):
        return self.down(F.relu(self.up(x)).square())  # ReLU^2: sparse activations


class ProductKeyMemory(nn.Module):
    """The Library: n_sub^2 value slots, each lookup costs O(n_sub) and reads only `topk` rows."""

    def __init__(self, d, n_sub=256, dk=128, heads=2, topk=16):
        super().__init__()
        self.n_sub, self.heads, self.topk, self.half = n_sub, heads, topk, dk // 2
        self.query = nn.Linear(d, heads * dk, bias=False)
        self.qnorm = nn.LayerNorm(dk // 2, elementwise_affine=False)
        self.keys = nn.Parameter(torch.randn(heads, 2, n_sub, dk // 2) / math.sqrt(dk // 2))
        self.values = nn.Embedding(n_sub * n_sub, d)
        nn.init.normal_(self.values.weight, std=d ** -0.5)

    def forward(self, x):
        B, T, D = x.shape
        q = self.qnorm(self.query(x).view(B, T, self.heads, 2, self.half))
        s = torch.einsum("bthcd,hcnd->bthcn", q, self.keys)  # (B,T,H,2,n_sub)
        s1, i1 = s[..., 0, :].topk(self.topk, -1)
        s2, i2 = s[..., 1, :].topk(self.topk, -1)
        comb = (s1.unsqueeze(-1) + s2.unsqueeze(-2)).flatten(-2)
        idx = (i1.unsqueeze(-1) * self.n_sub + i2.unsqueeze(-2)).flatten(-2)
        best, pos = comb.topk(self.topk, -1)
        idx = idx.gather(-1, pos)
        w = best.softmax(-1).unsqueeze(-1)
        return (self.values(idx) * w).sum(-2).sum(-2)  # sum over slots, then heads

    def bytes_read_per_token(self, d):
        keys = self.heads * 2 * self.n_sub * self.half  # int8 sub-keys
        query = self.query.weight.numel()  # int8 query projection
        rows = self.heads * self.topk * d * 2  # fp16 value rows (could be int8 on device)
        return keys + query + rows


class Block(nn.Module):
    def __init__(self, d, heads, lin, memory=None):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.attn = Attention(d, heads, lin)
        self.mlp = MLP(d, 4 * d, lin)
        self.memory = memory
        if memory is not None:
            self.n3 = RMSNorm(d)

    def forward(self, x):
        x = x + self.attn(self.n1(x))
        x = x + self.mlp(self.n2(x))
        if self.memory is not None:
            x = x + self.memory(self.n3(x))
        return x


class Model(nn.Module):
    def __init__(self, d, layers, loops=1, ternary=False, memory=False, heads=4):
        super().__init__()
        lin = BitLinear if ternary else nn.Linear
        self.ternary, self.loops = ternary, loops
        self.emb = nn.Embedding(IN_VOCAB, d)
        self.pos = nn.Parameter(torch.zeros(SEQ, d))
        self.mem = ProductKeyMemory(d) if memory else None
        self.blocks = nn.ModuleList(
            [Block(d, heads, lin, self.mem if (memory and i == layers - 1) else None) for i in range(layers)]
        )
        self.norm = RMSNorm(d)
        self.head = nn.Linear(d, OUT_VOCAB, bias=False)
        self.d = d

    def forward(self, x):
        h = self.emb(x) + self.pos
        for _ in range(self.loops):  # shared weights reused every loop
            for b in self.blocks:
                h = b(h)
        return self.head(self.norm(h[:, -1]))

    def bytes_read_per_token(self):
        """Weight bytes that must be streamed from memory to produce one token."""
        core = 0
        for b in self.blocks:
            for name, p in b.named_parameters():
                if name.startswith("memory."):
                    continue
                if p.dim() == 2:
                    core += p.numel() * (0.2 if self.ternary else 2)  # 1.58 bits packed ~ 0.2 bytes
                else:
                    core += p.numel() * 2
        mem = self.mem.bytes_read_per_token(self.d) if self.mem else 0
        head = self.head.weight.numel() * 2
        emb = self.d * 2
        return int(self.loops * (core + mem) + head + emb)

    def stored_params(self):
        return sum(p.numel() for p in self.parameters())


CONFIGS = {
    # standard dense fp16 transformer, ~3M params
    "dense_big": dict(d=256, layers=4),
    # dense fp16 transformer sized to match core_library's bytes-per-token
    "dense_matched": dict(d=160, layers=1),
    # the proposal: one ternary block looped 2x + product-key library (65k slots)
    "core_library": dict(d=256, layers=1, loops=2, ternary=True, memory=True),
}


# ---------------------------------------------------------------- train / eval

@torch.no_grad()
def accuracy(model, x, y, dev, bs=4096):
    model.eval()
    hit = 0
    for i in range(0, len(x), bs):
        hit += (model(x[i : i + bs].to(dev)).argmax(-1) == y[i : i + bs].to(dev)).sum().item()
    model.train()
    return hit / len(x)


def run(name, data, steps, dev, bs=1024, seed=0):
    torch.manual_seed(seed)
    (x1, y1), (x2, y2), (x2t, y2t) = data
    xtr, ytr = torch.cat([x1, x2]).to(dev), torch.cat([y1, y2]).to(dev)
    model = Model(**CONFIGS[name]).to(dev)

    mem_params = list(model.mem.values.parameters()) if model.mem else []
    mem_ids = {id(p) for p in mem_params}
    other = [p for p in model.parameters() if id(p) not in mem_ids]
    groups = [{"params": other, "lr": 2e-3}]
    if mem_params:
        groups.append({"params": mem_params, "lr": 1e-2, "weight_decay": 0.0})
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[g["lr"] for g in groups], total_steps=steps, pct_start=0.05
    )

    print(f"\n== {name}: {model.stored_params()/1e6:.2f}M stored params, "
          f"{model.bytes_read_per_token()/1e3:.0f} KB read/token")
    t0 = time.time()
    for step in range(1, steps + 1):
        i = torch.randint(0, len(xtr), (bs,), device=dev)
        loss = F.cross_entropy(model(xtr[i]), ytr[i])
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()
        if step % max(1, steps // 10) == 0 or step == steps:
            print(f"  step {step:6d}  loss {loss.item():.3f}  "
                  f"1hop {accuracy(model, x1, y1, dev):.3f}  "
                  f"2hop-train {accuracy(model, x2, y2, dev):.3f}  "
                  f"2hop-heldout {accuracy(model, x2t, y2t, dev):.3f}  "
                  f"({time.time()-t0:.0f}s)", flush=True)

    return {
        "model": name,
        "stored_params_M": round(model.stored_params() / 1e6, 2),
        "KB_read_per_token": round(model.bytes_read_per_token() / 1e3, 1),
        "acc_1hop_facts": accuracy(model, x1, y1, dev),
        "acc_2hop_train": accuracy(model, x2, y2, dev),
        "acc_2hop_heldout": accuracy(model, x2t, y2t, dev),
        "train_seconds": round(time.time() - t0),
    }


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--entities", type=int, default=30000)
    ap.add_argument("--steps", type=int, default=20000)
    ap.add_argument("--models", default="dense_big,dense_matched,core_library")
    ap.add_argument("--out", default="results/results.json")
    args = ap.parse_args()

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    data = build_sets(make_world(args.entities))
    print(f"device={dev}  facts={len(data[0][0])}  2hop-train={len(data[1][0])}  2hop-heldout={len(data[2][0])}")

    results = [run(m, data, args.steps, dev) for m in args.models.split(",")]
    print("\n" + json.dumps(results, indent=2))
    with open(args.out, "w") as f:
        json.dump(results, f, indent=2)

Writing src/experiment.py


In [4]:
%%writefile src/multihop.py
"""
Multi-hop reasoning: "who is X's mother's brother?" and much longer chains.

World: people with 7 random relations each (links are random, so no shortcuts: every hop must be followed).
Query:  X r1 r2 ... rk Q
  direct: -> answer person only
  cot:    -> r1 y1 r2 y2 ... rk yk END     (writes every intermediate person: chain-of-thought)
  tag:    X t1 r1 t2 r2 ... Q -> t1 r1 y1 t2 r2 y2 ... END
          (each step carries a random unique tag; the next step is found by looking up the current tag
           in the question instead of counting, so it does not depend on chain length)

Tests:
  facts         single relations, everyone
  in_dist       2-4 hop chains, training people
  new_people    2-4 hop chains starting from people never used in chain training
  unseen_combo  chains containing "mother -> brother", which never appears in training
  longer        (max+1)-(max+2) hop chains, where max is the longest training chain (default 4)
  much_longer   (max+3)-(max+4) hop chains
"""
import argparse, json, os, random, time
import torch
import torch.nn as nn
import torch.nn.functional as F

import experiment as base
from experiment import MLP, BitLinear, ProductKeyMemory, RMSNorm

RELS = ["mother", "father", "brother", "sister", "spouse", "boss", "friend"]
BANNED = ("mother", "brother")  # "mother's brother" is never trained
N_FIRST = N_LAST = 300
PAD, Q, END = 0, 1, 2
REL = {r: 3 + i for i, r in enumerate(RELS)}
FIRST0 = 3 + len(RELS)
LAST0 = FIRST0 + N_FIRST
VOCAB = LAST0 + N_LAST
N_TAGS = 12
TAG0 = VOCAB  # tag tokens come after the original vocab so older checkpoints still load
VOCAB_TAG = VOCAB + N_TAGS
SEQ = 80
IGNORE = -100

# ---------------------------------------------------------------- world


class World:
    def __init__(self, n, seed=0):
        rng = random.Random(seed)
        self.names = rng.sample([(f, l) for f in range(N_FIRST) for l in range(N_LAST)], n)
        self.link = {r: [rng.randrange(n) for _ in range(n)] for r in RELS}
        order = list(range(n))
        rng.shuffle(order)
        self.held = order[: n // 5]
        self.train_people = order[n // 5 :]
        self.n = n

    def tok(self, e):
        return [FIRST0 + self.names[e][0], LAST0 + self.names[e][1]]

    def example(self, start, rels, mode, tags=None):
        """Returns (tokens, targets, answer_tokens)."""
        if mode == "tag":
            steps = [[TAG0 + t, REL[r]] for t, r in zip(tags, rels)]
        else:
            steps = [[REL[r]] for r in rels]
        prompt = self.tok(start) + sum(steps, []) + [Q]
        e, out = start, []
        for r, step in zip(rels, steps):
            e = self.link[r][e]
            out += (step if mode != "direct" else []) + self.tok(e)
        if mode != "direct":
            out += [END]
        elif len(rels) > 1:
            out = self.tok(e)
        tokens = prompt + out
        targets = [IGNORE] * (len(prompt) - 1) + out + [IGNORE]
        return tokens, targets, self.tok(e)


def has_banned(rels):
    return any(a == BANNED[0] and b == BANNED[1] for a, b in zip(rels, rels[1:]))


def random_rels(rng, k, allow_banned=False, need_banned=False):
    while True:
        rels = [rng.choice(RELS) for _ in range(k)]
        b = has_banned(rels)
        if need_banned and not b:
            if k >= 2:
                i = rng.randrange(k - 1)
                rels[i], rels[i + 1] = BANNED
                return rels
            continue
        if b and not allow_banned and not need_banned:
            continue
        return rels


def make(world, rng, start, rels, mode):
    tags = rng.sample(range(N_TAGS), len(rels)) if mode == "tag" else None
    return world.example(start, rels, mode, tags)


def pad(rows):
    L = max(len(t) for t, _, _ in rows)
    x = torch.full((len(rows), L), PAD)
    y = torch.full((len(rows), L), IGNORE)
    for i, (t, tg, _) in enumerate(rows):
        x[i, : len(t)] = torch.tensor(t)
        y[i, : len(tg)] = torch.tensor(tg)
    return x, y


def train_batch(world, rng, bs, mode, max_hops=4):
    rows = []
    for _ in range(bs):
        if rng.random() < 0.5:  # 1-hop facts, everyone (this is the knowledge base)
            rows.append(make(world, rng, rng.randrange(world.n), [rng.choice(RELS)], mode))
        else:  # chains, training people only, never "mother brother"
            rows.append(make(world, rng, rng.choice(world.train_people), random_rels(rng, rng.randint(2, max_hops)), mode))
    return pad(rows)


def eval_sets(world, mode, n=1000, seed=123, max_hops=4):
    rng = random.Random(seed)
    tp, hp = world.train_people, world.held
    mk = lambda f: [f() for _ in range(n)]
    ex = lambda start, rels: make(world, rng, start, rels, mode)
    return {
        "facts": mk(lambda: ex(rng.randrange(world.n), [rng.choice(RELS)])),
        "in_dist": mk(lambda: ex(rng.choice(tp), random_rels(rng, rng.randint(2, max_hops)))),
        "new_people": mk(lambda: ex(rng.choice(hp), random_rels(rng, rng.randint(2, max_hops)))),
        "unseen_combo": mk(lambda: ex(rng.choice(tp), random_rels(rng, rng.randint(2, max_hops), need_banned=True))),
        "longer": mk(lambda: ex(rng.choice(tp), random_rels(rng, rng.randint(max_hops + 1, max_hops + 2)))),
        "much_longer": mk(lambda: ex(rng.choice(tp), random_rels(rng, rng.randint(max_hops + 3, max_hops + 4)))),
    }

# ---------------------------------------------------------------- model (RoPE attention so position is relative)


def rope(x):
    T, D = x.shape[-2], x.shape[-1]
    inv = 1.0 / (10000 ** (torch.arange(0, D, 2, device=x.device).float() / D))
    ang = torch.arange(T, device=x.device).float()[:, None] * inv[None]
    cos, sin = ang.cos(), ang.sin()
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).flatten(-2)


class RopeAttention(nn.Module):
    def __init__(self, d, heads, lin, rope=True):
        super().__init__()
        self.h, self.rope = heads, rope
        self.qkv = lin(d, 3 * d, bias=False)
        self.o = lin(d, d, bias=False)

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        if self.rope:
            q, k = rope(q), rope(k)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o(y.transpose(1, 2).reshape(B, T, D))


class Block(nn.Module):
    def __init__(self, d, heads, lin, memory=None, rope=True):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.attn = RopeAttention(d, heads, lin, rope)
        self.mlp = MLP(d, 4 * d, lin)
        self.memory = memory
        if memory is not None:
            self.n3 = RMSNorm(d)

    def forward(self, x):
        x = x + self.attn(self.n1(x))
        x = x + self.mlp(self.n2(x))
        if self.memory is not None:
            x = x + self.memory(self.n3(x))
        return x


class Model(nn.Module):
    def __init__(self, d, layers, loops=1, ternary=False, memory=False, heads=4, vocab=VOCAB, rope=True):
        super().__init__()
        lin = BitLinear if ternary else nn.Linear
        self.ternary, self.loops, self.d = ternary, loops, d
        self.emb = nn.Embedding(vocab, d)
        self.mem = ProductKeyMemory(d) if memory else None
        self.blocks = nn.ModuleList(
            [Block(d, heads, lin, self.mem if (memory and i == layers - 1) else None, rope) for i in range(layers)]
        )
        self.norm = RMSNorm(d)
        self.head = nn.Linear(d, vocab, bias=False)

    def forward(self, x):
        h = self.emb(x)
        for _ in range(self.loops):
            for b in self.blocks:
                h = b(h)
        return self.head(self.norm(h))

    bytes_read_per_token = base.Model.bytes_read_per_token
    stored_params = base.Model.stored_params


CONFIGS = {
    "dense_big": dict(d=256, layers=4),
    "dense_matched": dict(d=160, layers=1),
    "core_library": dict(d=256, layers=1, loops=2, ternary=True, memory=True),
    "core_library_l4": dict(d=256, layers=1, loops=4, ternary=True, memory=True),
    # no position encoding: attention can only find things by content, not by distance
    "core_library_nope": dict(d=256, layers=1, loops=2, ternary=True, memory=True, rope=False),
}

# ---------------------------------------------------------------- eval: the model must GENERATE its own answer


@torch.no_grad()
def accuracy(model, rows, dev, bs=500):
    model.eval()
    hit = 0
    for i in range(0, len(rows), bs):
        chunk = rows[i : i + bs]
        plen = [t.index(Q) + 1 for t, _, _ in chunk]
        glen = [len(t) - p for (t, _, _), p in zip(chunk, plen)]
        L = max(p + g for p, g in zip(plen, glen))
        x = torch.full((len(chunk), L), PAD, device=dev)
        for j, (t, _, _) in enumerate(chunk):
            x[j, : plen[j]] = torch.tensor(t[: plen[j]])
        pos = torch.tensor(plen, device=dev)
        for _ in range(max(glen)):  # greedy decoding, one token at a time
            nxt = model(x)[torch.arange(len(chunk)), pos - 1].argmax(-1)
            x[torch.arange(len(chunk)), pos.clamp(max=L - 1)] = nxt
            pos = pos + 1
        for j, (t, _, ans) in enumerate(chunk):
            gen = x[j, plen[j] : plen[j] + glen[j]].tolist()
            final = gen[-3:-1] if t[-1] == END else gen[-2:]  # cot ends "... yk END"
            hit += final == ans
    model.train()
    return hit / len(rows)


def evaluate(model, sets, dev):
    return {k: accuracy(model, v, dev) for k, v in sets.items()}

# ---------------------------------------------------------------- train


def run(mode, name, world, steps, dev, bs=512, seed=0, save_dir=None, eval_n=1000, max_hops=4):
    torch.manual_seed(seed)
    rng = random.Random(seed)
    sets = eval_sets(world, mode, n=eval_n, max_hops=max_hops)
    vocab = VOCAB_TAG if mode == "tag" else VOCAB
    model = Model(**CONFIGS[name], vocab=vocab).to(dev)

    mem_params = list(model.mem.values.parameters()) if model.mem else []
    mem_ids = {id(p) for p in mem_params}
    groups = [{"params": [p for p in model.parameters() if id(p) not in mem_ids], "lr": 2e-3}]
    if mem_params:
        groups.append({"params": mem_params, "lr": 1e-2, "weight_decay": 0.0})
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[g["lr"] for g in groups], total_steps=steps, pct_start=0.05
    )

    print(f"\n== {mode}:{name}  {model.stored_params()/1e6:.2f}M stored, "
          f"{model.bytes_read_per_token()/1e3:.0f} KB read/token", flush=True)
    t0 = time.time()
    for step in range(1, steps + 1):
        x, y = (t.to(dev) for t in train_batch(world, rng, bs, mode, max_hops))
        loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten(), ignore_index=IGNORE)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()
        if step % max(1, steps // 10) == 0 or step == steps:
            r = evaluate(model, sets, dev)
            print(f"  step {step:6d}  loss {loss.item():.3f}  " +
                  "  ".join(f"{k} {v:.3f}" for k, v in r.items()) + f"  ({time.time()-t0:.0f}s)", flush=True)

    r = evaluate(model, sets, dev)
    if save_dir:
        torch.save({"mode": mode, "config": name, "people": world.n, "vocab": vocab, "max_hops": max_hops,
                    "state": model.state_dict()}, f"{save_dir}/{mode}_{name}_h{max_hops}.pt")
    return {"mode": mode, "model": name, "max_train_hops": max_hops, "KB_read_per_token": round(model.bytes_read_per_token() / 1e3, 1),
            **{f"acc_{k}": v for k, v in r.items()}, "train_seconds": round(time.time() - t0)}


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--people", type=int, default=5000)
    ap.add_argument("--steps", type=int, default=8000)
    ap.add_argument("--runs", default="cot:core_library,direct:core_library_l4,cot:dense_matched")
    ap.add_argument("--out", default="results/multihop_results.json")
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    ap.add_argument("--save-dir", default="checkpoints")
    ap.add_argument("--eval-n", type=int, default=1000, help="test questions per set")
    ap.add_argument("--max-hops", type=int, default=4, help="longest chain used in training")
    ap.add_argument("--batch-size", type=int, default=512)
    args = ap.parse_args()

    os.makedirs(args.save_dir, exist_ok=True)
    world = World(args.people)
    results = []
    for spec in args.runs.split(","):
        mode, name = spec.split(":")
        results.append(run(mode, name, world, args.steps, args.device, save_dir=args.save_dir, eval_n=args.eval_n, max_hops=args.max_hops, bs=args.batch_size))
        with open(args.out, "w") as f:
            json.dump(results, f, indent=2)
    print("\n" + json.dumps(results, indent=2))

Writing src/multihop.py


In [5]:
%%writefile src/train_lm.py
"""
Train the Core + Library language model on the token files from prepare_data.py.

  .venv\Scripts\python src\train_lm.py                    # starts, or resumes from checkpoints/lm/latest.pt
  .venv\Scripts\python src\train_lm.py --tokens 3e9       # total training tokens (default 3B)
  torchrun --nproc_per_node=2 src/train_lm.py ...          # several GPUs (e.g. Kaggle T4 x2): data parallel

Writes:
  logs/lm_train.log       progress: loss, tokens/s, ETA
  logs/lm_samples.txt     sample stories every 30 min, so you can read what it is learning
  checkpoints/lm/         latest.pt every 15 min (safe to stop and resume any time), plus a copy per billion tokens
"""
import argparse, contextlib, math, os, sys, time
import numpy as np
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(__file__))
from experiment import BitLinear, ProductKeyMemory, RMSNorm
from multihop import Block

DATA = "data/tokens"
CKPT = "checkpoints/lm"
SOURCES = {"pg19": 0.5, "literotica": 0.5}  # sampling mix

CONFIGS = {
    # ~35M stored params; per token reads the ternary core twice (~2.8 MB) plus 32 Library rows
    "core_library": dict(d=384, layers=4, loops=2, heads=6, ternary=True, memory=True),
    # dense fp16 baseline with a similar per-token read budget
    "dense": dict(d=384, layers=2, loops=1, heads=6, ternary=False, memory=False),
    "tiny": dict(d=64, layers=1, loops=1, heads=2, ternary=True, memory=True),  # for smoke tests
}


class LM(nn.Module):
    def __init__(self, vocab, d, layers, loops, heads, ternary, memory):
        super().__init__()
        lin = BitLinear if ternary else nn.Linear
        self.loops, self.ternary, self.d = loops, ternary, d
        self.emb = nn.Embedding(vocab, d)
        nn.init.normal_(self.emb.weight, std=0.02)
        self.mem = ProductKeyMemory(d) if memory else None
        self.blocks = nn.ModuleList(
            [Block(d, heads, lin, self.mem if (memory and i == layers // 2) else None) for i in range(layers)]
        )
        self.norm = RMSNorm(d)

    def forward(self, x):
        h = self.emb(x)
        for _ in range(self.loops):
            for b in self.blocks:
                h = b(h)
        return F.linear(self.norm(h), self.emb.weight)  # output layer shares the embedding


# ---------------------------------------------------------------- data


class Stream:
    def __init__(self, split, seq, seed, data_dir=DATA):
        self.seq, self.rng = seq, np.random.default_rng(seed)
        self.data = {s: np.memmap(f"{data_dir}/{s}_{split}.bin", dtype=np.uint16, mode="r") for s in SOURCES}
        self.names = list(SOURCES)
        self.p = np.array([SOURCES[s] for s in self.names]) / sum(SOURCES.values())

    def batch(self, bs, source=None):
        src = source or self.names[self.rng.choice(len(self.names), p=self.p)]
        arr = self.data[src]
        starts = self.rng.integers(0, len(arr) - self.seq - 1, bs)
        chunk = np.stack([arr[i : i + self.seq + 1] for i in starts]).astype(np.int64)
        t = torch.from_numpy(chunk)
        return t[:, :-1], t[:, 1:]


# ---------------------------------------------------------------- generation


@torch.no_grad()
def generate(model, tok, prompt, dev, n=200, temp=0.8, top_k=50, seq=512):
    was = model.training
    model.eval()
    ids = torch.tensor([tok.encode(prompt).ids], device=dev)
    eot = tok.token_to_id("<|endoftext|>")
    for _ in range(n):
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev.startswith("cuda")):
            logits = model(ids[:, -seq:])[0, -1].float() / temp
        v, i = logits.topk(top_k)
        nxt = i[torch.multinomial(v.softmax(-1), 1)]
        if nxt.item() == eot:
            break
        ids = torch.cat([ids, nxt[None]], 1)
    model.train(was)
    return tok.decode(ids[0].tolist())


PROMPTS = [
    "Title: The Lighthouse Keeper\n\nThe storm had been building since noon, and",
    "Title: Midnight in Paris\nCategory: Romance\n\nShe hadn't expected to see him again, not here,",
    "Title: The Inheritance\n\nIt was on a grey morning in November that",
]

# ---------------------------------------------------------------- train


@torch.no_grad()
def evaluate(model, val, bs, dev, batches=20):
    model.eval()
    out = {}
    for s in SOURCES:
        tot = 0.0
        for _ in range(batches):
            x, y = (t.to(dev) for t in val.batch(bs, s))
            with torch.autocast("cuda", dtype=torch.float16, enabled=dev.startswith("cuda")):
                tot += F.cross_entropy(model(x).float().flatten(0, 1), y.flatten()).item()
        out[s] = tot / batches
    model.train()
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="core_library", choices=CONFIGS)
    ap.add_argument("--tokens", type=float, default=3e9)
    ap.add_argument("--seq", type=int, default=512)
    ap.add_argument("--bs", type=int, default=16)
    ap.add_argument("--accum", type=int, default=4)
    ap.add_argument("--lr", type=float, default=1.5e-3)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    ap.add_argument("--ckpt-dir", default=CKPT)
    ap.add_argument("--save-min", type=float, default=15)
    ap.add_argument("--sample-min", type=float, default=30)
    ap.add_argument("--eval-every", type=int, default=500)
    ap.add_argument("--log", default="logs/lm_train.log")
    ap.add_argument("--data-dir", default=DATA)
    ap.add_argument("--tokenizer", default="data/tokenizer.json")
    ap.add_argument("--max-hours", type=float, default=0, help="save and stop after this long (Kaggle 12 h limit)")
    args = ap.parse_args()

    ddp = int(os.environ.get("WORLD_SIZE", "1")) > 1
    if ddp:  # launched by torchrun: one process per GPU, each trains on different batches
        gpu = torch.cuda.is_available() and dist.is_nccl_available()
        dist.init_process_group("nccl" if gpu else "gloo")  # gloo: CPU-only test runs (no NCCL on Windows)
        rank, world = dist.get_rank(), dist.get_world_size()
        local = int(os.environ["LOCAL_RANK"])
        if gpu:
            torch.cuda.set_device(local)
        dev = f"cuda:{local}" if gpu else "cpu"
    else:
        rank, world, dev = 0, 1, args.device
    master = rank == 0

    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(args.tokenizer)
    torch.manual_seed(0)
    model = LM(tok.get_vocab_size(), **CONFIGS[args.config]).to(dev)

    mem_ids = {id(p) for p in model.mem.values.parameters()} if model.mem else set()
    groups = [{"params": [p for p in model.parameters() if id(p) not in mem_ids], "lr": args.lr, "weight_decay": 0.1}]
    if mem_ids:
        groups.append({"params": list(model.mem.values.parameters()), "lr": args.lr * 4, "weight_decay": 0.0})
    for g in groups:
        g["base_lr"] = g["lr"]
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.95))
    scaler = torch.amp.GradScaler("cuda", enabled=dev.startswith("cuda"))

    tokens_per_step = args.bs * args.seq * args.accum * world
    total_steps = int(args.tokens // tokens_per_step)
    warmup = min(1000, total_steps // 20 + 1)
    os.makedirs(args.ckpt_dir, exist_ok=True)
    os.makedirs(os.path.dirname(args.log), exist_ok=True)
    latest = os.path.join(args.ckpt_dir, "latest.pt")
    step = 0
    if os.path.exists(latest):
        ck = torch.load(latest, map_location=dev)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        scaler.load_state_dict(ck["scaler"])
        step = ck["step"]
        if master:
            print(f"resumed from step {step}")
    raw = model
    if ddp:
        model = torch.nn.parallel.DistributedDataParallel(model, device_ids=[local] if dev != "cpu" else None)

    def save(path):
        torch.save({"model": raw.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                    "step": step, "config": args.config, "vocab": tok.get_vocab_size()}, path + ".tmp")
        os.replace(path + ".tmp", path)

    log = open(args.log, "a", encoding="utf-8") if master else None
    samples = open(os.path.join(os.path.dirname(args.log), "lm_samples.txt"), "a", encoding="utf-8") if master else None

    def say(msg):
        if master:
            print(msg, flush=True)
            log.write(msg + "\n")
            log.flush()

    n_params = sum(p.numel() for p in model.parameters())
    say(f"== {args.config}: {n_params/1e6:.1f}M params | {total_steps} steps x {tokens_per_step} tokens "
        f"= {total_steps * tokens_per_step / 1e9:.2f}B tokens | start step {step}")

    train = Stream("train", args.seq, seed=step + 1 + 1000 * rank, data_dir=args.data_dir)
    val = Stream("val", args.seq, seed=1234, data_dir=args.data_dir)
    t_save = t_sample = time.time()
    t0, tok0 = time.time(), step * tokens_per_step
    milestone = (step * tokens_per_step) // 1_000_000_000

    t_start = time.time()
    while step < total_steps:
        if args.max_hours and time.time() - t_start > args.max_hours * 3600:
            if master:
                save(latest)
                say(f"time limit reached at step {step}: saved, resume to continue")
            break
        lr_scale = min(1.0, (step + 1) / warmup) * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * step / total_steps)))
        for g in opt.param_groups:
            g["lr"] = g["base_lr"] * lr_scale
        for micro in range(args.accum):
            x, y = (t.to(dev, non_blocking=True) for t in train.batch(args.bs))
            sync = not ddp or micro == args.accum - 1  # share gradients between GPUs once per step
            with (contextlib.nullcontext() if sync else model.no_sync()):
                with torch.autocast("cuda", dtype=torch.float16, enabled=dev.startswith("cuda")):
                    loss = F.cross_entropy(model(x).float().flatten(0, 1), y.flatten())
                scaler.scale(loss / args.accum).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        step += 1

        if not master:
            continue
        if step % 50 == 0 or step == total_steps:
            done = step * tokens_per_step
            rate = (done - tok0) / (time.time() - t0)
            eta_h = (total_steps * tokens_per_step - done) / max(rate, 1) / 3600
            mem = torch.cuda.max_memory_reserved() / 2**30 if dev.startswith("cuda") else 0
            say(f"step {step:7d}  tokens {done/1e6:8.1f}M  loss {loss.item():.3f}  {rate:,.0f} tok/s  "
                f"ETA {eta_h:5.1f} h  gpu-mem {mem:.2f} GB  {time.strftime('%H:%M')}")
        if step % args.eval_every == 0 or step == total_steps:
            v = evaluate(raw, val, args.bs, dev)
            say("  val loss  " + "  ".join(f"{k} {l:.3f} (ppl {math.exp(l):.0f})" for k, l in v.items()))
        if time.time() - t_save > args.save_min * 60 or step == total_steps:
            save(latest)
            t_save = time.time()
            m = (step * tokens_per_step) // 1_000_000_000
            if m > milestone:
                save(os.path.join(args.ckpt_dir, f"lm_{m}B.pt"))
                milestone = m
        if time.time() - t_sample > args.sample_min * 60 or step == total_steps:
            samples.write(f"\n===== step {step}  ({step * tokens_per_step / 1e6:.0f}M tokens, {time.strftime('%H:%M')}) =====\n")
            for p in PROMPTS:
                samples.write("\n" + generate(raw, tok, p, dev) + "\n---\n")
            samples.flush()
            t_sample = time.time()
    if master and step >= total_steps:
        save(os.path.join(args.ckpt_dir, "final.pt"))
        say("DONE")
    if ddp:
        dist.destroy_process_group()


if __name__ == "__main__":
    main()

Writing src/train_lm.py


In [6]:
import glob, os, shutil, subprocess

def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    return hits[0] if hits else None

OUT = "/kaggle/working/data"
prev_tok = find("data/tokenizer.json")
if prev_tok:  # continuing: reuse the data and checkpoint from the previous version
    src_dir = os.path.dirname(prev_tok)
    shutil.copy(prev_tok, f"{OUT}/tokenizer.json")
    shutil.copytree(f"{src_dir}/tokens", f"{OUT}/tokens", dirs_exist_ok=True)
    for rel in ("checkpoints/lm/latest.pt", "logs/lm_train.log", "logs/lm_samples.txt"):
        old = find(rel)
        if old and not os.path.exists(f"/kaggle/working/{rel}"):
            shutil.copy(old, f"/kaggle/working/{rel}")
    print("continuing: data reused, checkpoint", "found" if find("checkpoints/lm/latest.pt") else "not found")
HAVE_DATA = os.path.exists(f"{OUT}/tokens/pg19_train.bin")
print("prepared data present:", HAVE_DATA)

prepared data present: False


In [7]:
from huggingface_hub import snapshot_download

RAW = "/tmp/data"  # big temporary files stay out of /kaggle/working
if not HAVE_DATA:
    for name, repo in [("pg19", "emozilla/pg19"), ("literotica", "detain/literotica-stories")]:
        snapshot_download(repo_id=repo, repo_type="dataset", allow_patterns=["data/train-*.parquet"],
                          local_dir=f"{RAW}/{name}_hf", max_workers=8)
        os.makedirs(f"{RAW}/{name}", exist_ok=True)
        with open(f"{RAW}/{name}/download.log", "w") as log:
            for f in sorted(glob.glob(f"{RAW}/{name}_hf/data/train-*.parquet")):
                os.replace(f, f"{RAW}/{name}/{os.path.basename(f)}")
                log.write(f"{time.strftime('%H:%M:%S')} {os.path.basename(f)} exit=0\n")
        print(name, len(glob.glob(f"{RAW}/{name}/*.parquet")), "files")

    def run(*args):
        subprocess.run(["python", "src/prepare_data.py", "--data-dir", RAW, *args], check=True)

    run("--stage", "clean", "--workers", "4")
    shutil.rmtree(f"{RAW}/pg19"); shutil.rmtree(f"{RAW}/literotica")
    for d in glob.glob(f"{RAW}/*_hf"):
        shutil.rmtree(d)
    run("--stage", "tokenizer")
    run("--stage", "tokenize", "--max-tokens", "1.5e9")
    shutil.copytree(f"{RAW}/tokens", f"{OUT}/tokens", dirs_exist_ok=True)
    shutil.copy(f"{RAW}/tokenizer.json", f"{OUT}/tokenizer.json")
    shutil.copy(f"{RAW}/clean/report.json", f"{OUT}/report.json")
    print(open(f"{OUT}/report.json").read())
print(f"data ready after {(time.time() - T0) / 3600:.2f} h")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]

pg19 23 files


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 37 files:   0%|          | 0/37 [00:00<?, ?it/s]

literotica 37 files


... (2 outputs left out)

/kaggle/working/src/prepare_data.py:4: SyntaxWarning: invalid escape sequence '\S'
  .venv\Scripts\python src\prepare_data.py --stage clean          # data/clean/<source>/*.jsonl.gz + report





vocab 8192  |  test sentence: 76 chars -> 19 tokens
  roundtrip ok: True


/kaggle/working/src/prepare_data.py:4: SyntaxWarning: invalid escape sequence '\S'
  .venv\Scripts\python src\prepare_data.py --stage clean          # data/clean/<source>/*.jsonl.gz + report


  pg19: 1,502.7M train / 7.1M val tokens  (14,887 unique docs)
  literotica: 0.0M train / 0.0M val tokens  (0 unique docs)
{
  "pg19": {
    "read": 28602,
    "kept": 28595,
    "chars": 11241730490,
    "drop:too_short": 7
  },
  "literotica": {
    "read": 645621,
    "drop:minors": 645621,
    "drop:minors:text": 645621
  }
}
data ready after 1.04 h


In [8]:
HOURS = round(11.3 - (time.time() - T0) / 3600, 2)  # leave time to save the output
print("training for up to", HOURS, "h")
!torchrun --nproc_per_node=2 src/train_lm.py --tokens 3e9 --bs 32 --accum 2 \
    --data-dir {OUT}/tokens --tokenizer {OUT}/tokenizer.json \
    --ckpt-dir /kaggle/working/checkpoints/lm --log /kaggle/working/logs/lm_train.log --max-hours {HOURS}

training for up to 10.26 h
W1031 13:48:34.162000 540 torch/distributed/run.py:851] 
W1031 13:48:34.162000 540 torch/distributed/run.py:851] *****************************************
W1031 13:48:34.162000 540 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1031 13:48:34.162000 540 torch/distributed/run.py:851] *****************************************
/kaggle/working/src/train_lm.py:4: SyntaxWarning: invalid escape sequence '\S'
  .venv\Scripts\python src\train_lm.py                    # starts, or resumes from checkpoints/lm/latest.pt
/kaggle/working/src/train_lm.py:4: SyntaxWarning: invalid escape sequence '\S'
  .venv\Scripts\python src\train_lm.py                    # starts, or resumes from checkpoints/lm/latest.pt
== core_library: 35.6M params | 45776 steps x 65536 tokens = 3.00B tokens | st

In [9]:
!tail -5 /kaggle/working/logs/lm_train.log
!tail -40 /kaggle/working/logs/lm_samples.txt

== core_library: 35.6M params | 45776 steps x 65536 tokens = 3.00B tokens | start step 0
